This code aims to take a shapefile of station locations and measuements and removes a subset of stations at random to rerun in Greg's code. This could be implemented in some sort of potential cross validation.

In [1]:
import sys
import os as os

import geopandas as gpd
import numpy as np
from numpy.polynomial import Polynomial
import psycopg2
from netCDF4 import Dataset

from tqdm import tqdm
from multiprocessing import Pool
import statsmodels.api as sm
from scipy import stats
# from scipy import optimize

import cartopy.crs as ccrs
from cartopy.feature import NaturalEarthFeature as cfNEF

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import patches
#import matplotlib.patches as patches
import matplotlib.patheffects as path_effects
from matplotlib.lines import Line2D

import rasterio
import xarray as xr

import random
import subprocess
import netCDF4
import shutil

In [4]:
# I do not see any UMRB Stations
unique_station_ty = gdf['STATION_TY'].unique()
print(unique_station_ty)

['ASOS' 'COOPB' 'COOPABC' 'ASOS, AWIPS, COOPABC, CRS, UA, WSR88D' 'COOPAB'
 'MESO-ST' 'COOPB, GOES' 'COCORAHS' 'UCOOP' 'COOPBC' 'GOES' 'WSF'
 'ASOS, ASR11, COOPABC' 'ASOS, AWIPS, COOPAB, CRS, UA, WSR88D' 'SNOTEL'
 'WSR88D' 'COOPC' 'COOPAB, GOES, RFCSIM' 'OTHER'
 'AWIPS, COOPBC, CRS, WSR88D' 'COOPA' 'ASOS, AWOS, COOPABC' 'APRSWXNET'
 'AMOS, ASOS, AWIPS, COOPAB, CRS, WSR88D' 'COOPAC' 'ASOS, COOPAB'
 'UNKNOWN' 'ASOS, COOPB' 'COOPAB, RFCSIM' 'ARC' 'ASOS, COOPABC' 'RAMOS'
 'ASOS, COOPABC, SAWRS' 'NWS_OFFICIAL'
 'ASOS, AWIPS, COOPAB, CRS, TDWR, WSR88D' 'ASOS, COOPAB, UA' 'ICAO'
 'COOPAB, GOES' 'AWIPS, CRS, WSR88D' 'AWIPS, COOPAB, CRS, UA, WSR88D'
 'MESO-UMRB' 'ASOS, COOPAB, TDWR' 'MNGAGE' 'AWIPS, COOPAB, CRS, WSR88D'
 'ASOS, AWIPS, COOPB, CRS, WSR88D' 'GOES, NWIS' 'AHOS, COOPAB, GOES'
 'COOPB, GOES, RFCSIM' 'COOPAB, GOES, NWIS' 'ASOS, COOPB, TDWR'
 'AWIPS, COOPBC, CRS, UA, WSR88D' 'BASIC, COOPABC' None]


In [28]:
# Load the shapefile
shapefile_path = "C:\\Repos\\umrb-snodas\\examples\\snodas_assim_20221112\\ssm1054_md_based_2022111112_2022111212_us.shp" 
gdf = gpd.read_file(shapefile_path)


# Filter out rows where latitude or longitude is NaN and keep only rows with 'SNOTEL' in 'STATION_TY'
gdf_filtered = gdf[gdf['STATION_TY'] == 'SNOTEL']

# List to store subsets
subsets_list = []

for x in range(1):
    
    gdf = gpd.read_file(shapefile_path)
    
    # Randomly select a subset of stations
    subset_size = 10
    if len(gdf_filtered) < subset_size:
        print("Error: Subset size is larger than the number of stations.")
    else:
        valid_indices = gdf_filtered.index.tolist()
        subset_indices = random.sample(valid_indices, subset_size)
        subset = gdf_filtered.loc[subset_indices]  # Use gdf_filtered instead of gdf

        # Append the subset DataFrame to the list
        subsets_list.append(subset)
        
        # Drop subsets from original gdf
        gdf = gdf.drop(subset_indices)

        # Write new shapefile to new folder - had to save twice to get correct shapefile name - Ideas???
        gdf.to_file(f"C:/Repos/umrb-snodas/examples/snodas_assim_20221112{x}")
        gdf.to_file(f"C:/Repos/umrb-snodas/examples/snodas_assim_20221112{x}/ssm1054_md_based_2022111112_2022111212_us{x}.shp")
        
        # Copy netCDF's from orgignal folder to new folder
        shutil.copy('C:\\Repos\\umrb-snodas\\examples\\snodas_assim_20221112\\ssm1054_2022111212.nc',f"C:/Repos/umrb-snodas/examples/snodas_assim_20221112{x}/ssm1054_2022111212.nc" )
        shutil.copy('C:\\Repos\\umrb-snodas\\examples\\snodas_assim_20221112\\ssm_process_region_2022111112_2022111212_swe_us.nc',f"C:/Repos/umrb-snodas/examples/snodas_assim_20221112{x}/ssm_process_region_2022111112_2022111212_swe_us{x}.nc" )
        
        cmd = f"python ./snodas_idw.py -k -g -i 3 50 1 0 250 2022111112 2022111212 us{x} -f ../examples/snodas_assim_20221112{x}"
   
        subprocess.run(cmd, capture_output=True)
        
        print(subsets_list)



Notes for Paul:

The new folders being created need the netCDF Files within the original example files to run properly. I used the shutil.copy command to copy those netCDF files to the new folder that was created for each iteration with a new name saved to it that corresponds to the command line layout that it has to have to work. Also the code seemed to need the shapefile names just the way I changed it within the "cmd" above with the number of iteration it was at the end of the -d section and -f section. The folders that the shapefiles and all other files are saved in needed to be named the same as the shapefile in order to work. I had to save my code here within the src folder for it to work as well. Also I am not seeing any outputs being saved directly to the folder but I can get the shapefile to pop up on the screen so I know its working in some regard. Also the new folder seems to be creating a CPG File, not sure what that is.